# Hub Data Post-Processing

This notebook performs post-processing on scored hub data:
1. **Add HubName column** - Merge hub names from an external CSV file
2. **Display ranked tables** - Create summary tables by rank and hub type

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, HTML

# Setup paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

# Create directories if needed
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## Step 1: Load Scored Hub Data

Load the hub data from the scoring pipeline output.

In [ ]:
# Configure input file path
# Option 1: Load from results directory (output of scoring pipeline)
# scored_hubs_file = RESULTS_DIR / 'scored_hubs_YYYYMMDD.csv'

# Option 2: Load from notebooks directory (sample data)
scored_hubs_file = NOTEBOOKS_DIR / 'grouped_hubs_ready_for_scoring_21082025.csv'

# Load the data
df = pd.read_csv(scored_hubs_file, encoding='utf-8')

print(f"Loaded {len(df)} hubs from: {scored_hubs_file.name}")
print(f"\nColumns available: {len(df.columns)}")
print(df.columns.tolist())

## Step 2: Load Hub Names CSV

Load the external CSV file containing hub names mapped to group IDs.

**Expected format:**
```
group,HubName
0,Hub Name 1
1,Hub Name 2
...
```

In [ ]:
# Configure hub names file path
hub_names_file = DATA_DIR / 'hub_names.csv'

# Check if file exists
if hub_names_file.exists():
    hub_names_df = pd.read_csv(hub_names_file, encoding='utf-8')
    print(f"Loaded {len(hub_names_df)} hub names from: {hub_names_file.name}")
    print(f"\nColumns: {hub_names_df.columns.tolist()}")
    display(hub_names_df.head())
else:
    print(f"WARNING: Hub names file not found at: {hub_names_file}")
    print("\nTo use this feature, create a CSV file with columns:")
    print("  - group: The group ID (integer)")
    print("  - HubName: The human-readable hub name (string)")
    hub_names_df = None

## Step 3: Merge Hub Names

Add the HubName column to the main dataframe by joining on the group ID.

In [ ]:
def add_hub_names(df, hub_names_df, group_col='group', name_col='HubName'):
    """
    Merge hub names into the main dataframe.
    
    Args:
        df: Main hub dataframe
        hub_names_df: DataFrame with group IDs and hub names
        group_col: Column name for group ID (default: 'group')
        name_col: Column name for hub name (default: 'HubName')
    
    Returns:
        DataFrame with HubName column added
    """
    if hub_names_df is None:
        print("No hub names data available. Using address as fallback.")
        df['HubName'] = df['address'].apply(
            lambda x: str(x).split(',')[0] if pd.notna(x) else f"Hub {df.index}"
        )
        return df
    
    # Ensure group column is the same type in both dataframes
    df[group_col] = df[group_col].astype(int)
    hub_names_df[group_col] = hub_names_df[group_col].astype(int)
    
    # Merge on group column
    df = df.merge(
        hub_names_df[[group_col, name_col]], 
        on=group_col, 
        how='left'
    )
    
    # Fill missing names with address or group ID
    missing_mask = df[name_col].isna()
    if missing_mask.any():
        print(f"Warning: {missing_mask.sum()} hubs have no name assigned.")
        df.loc[missing_mask, name_col] = df.loc[missing_mask, 'address'].apply(
            lambda x: str(x).split(',')[0] if pd.notna(x) else None
        )
        # Final fallback to group ID
        still_missing = df[name_col].isna()
        df.loc[still_missing, name_col] = df.loc[still_missing, group_col].apply(
            lambda x: f"Hub {x}"
        )
    
    return df

# Add hub names
df = add_hub_names(df, hub_names_df)

print(f"\nHubName column added successfully!")
print(f"\nSample hub names:")
display(df[['group', 'HubName', 'address']].head(10))

## Step 4: Prepare Data for Display

Ensure ranking columns exist and prepare display columns.

In [ ]:
# Check for score column (from Monte Carlo scoring)
score_col = None
for col in ['Average_Simulated_Score', 'final_score', 'score']:
    if col in df.columns:
        score_col = col
        break

if score_col is None:
    print("WARNING: No score column found. Creating placeholder rankings.")
    df['Score'] = df['TotalDemand']  # Fallback to demand-based ranking
    score_col = 'Score'

# Add rankings if not present
if 'Overall_Rank' not in df.columns:
    df['Overall_Rank'] = df[score_col].rank(method='dense', ascending=False)

if 'Rank_within_HubType' not in df.columns and 'HubType' in df.columns:
    df['Rank_within_HubType'] = df.groupby('HubType')[score_col].rank(
        method='dense', ascending=False
    )

print(f"Score column: {score_col}")
print(f"Rankings added/verified.")

## Step 5: Display Tables by Rank

### 5.1 Overall Top Hubs (All Types)

In [ ]:
def create_display_table(df, columns, title, n_rows=20):
    """
    Create a formatted display table.
    
    Args:
        df: DataFrame to display
        columns: List of columns to include
        title: Table title
        n_rows: Number of rows to display
    
    Returns:
        Styled DataFrame for display
    """
    # Filter to available columns
    available_cols = [c for c in columns if c in df.columns]
    
    display_df = df[available_cols].head(n_rows).copy()
    
    # Format numeric columns
    for col in display_df.columns:
        if display_df[col].dtype in ['float64', 'float32']:
            if 'Rank' in col:
                display_df[col] = display_df[col].apply(lambda x: f"{int(x)}" if pd.notna(x) else "")
            elif 'Demand' in col:
                display_df[col] = display_df[col].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else "")
            else:
                display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "")
    
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    
    return display_df

# Define display columns
display_cols = [
    'Overall_Rank',
    'HubName',
    'HubType',
    'TotalDemand',
    score_col,
    'Num_Modes',
    'area'
]

# Sort by overall rank and display
df_sorted = df.sort_values('Overall_Rank')
top_hubs_table = create_display_table(
    df_sorted, 
    display_cols, 
    "TOP 20 HUBS (Overall Ranking)",
    n_rows=20
)
display(top_hubs_table)

### 5.2 Hubs by Type (Separate Tables)

In [ ]:
def display_hubs_by_type(df, hub_types=None, n_rows=10):
    """
    Display separate tables for each hub type.
    
    Args:
        df: Main hub dataframe
        hub_types: List of hub types to display (None = all)
        n_rows: Number of rows per table
    """
    if 'HubType' not in df.columns:
        print("HubType column not found.")
        return
    
    # Get unique hub types
    if hub_types is None:
        hub_types = df['HubType'].unique()
    
    # Define Hebrew-English type labels for display
    type_labels = {
        'ארצי': 'National (ארצי)',
        'מטרופוליני': 'Metropolitan (מטרופוליני)', 
        'עירוני': 'Local (עירוני)',
        'Regional': 'Regional/Metropolitan',
        'National': 'National',
        'Local': 'Local',
        'Train Station': 'Train Station',
        'Not Hub': 'Not Classified as Hub'
    }
    
    display_cols = [
        'Rank_within_HubType',
        'HubName',
        'TotalDemand',
        score_col,
        'Num_Modes',
        'area',
        'group'
    ]
    
    for hub_type in hub_types:
        # Filter for this hub type
        type_df = df[df['HubType'] == hub_type].copy()
        
        if len(type_df) == 0:
            continue
        
        # Sort by rank within type
        if 'Rank_within_HubType' in type_df.columns:
            type_df = type_df.sort_values('Rank_within_HubType')
        elif score_col in type_df.columns:
            type_df = type_df.sort_values(score_col, ascending=False)
        
        # Get display label
        label = type_labels.get(hub_type, hub_type)
        
        # Create and display table
        table = create_display_table(
            type_df,
            display_cols,
            f"{label} - {len(type_df)} hubs",
            n_rows=n_rows
        )
        display(table)
        print()  # Add spacing between tables

# Display tables by hub type
display_hubs_by_type(df, n_rows=10)

## Step 6: Summary Statistics

In [ ]:
def display_summary_statistics(df):
    """
    Display summary statistics by hub type.
    """
    print("\n" + "="*60)
    print("SUMMARY STATISTICS BY HUB TYPE")
    print("="*60)
    
    if 'HubType' not in df.columns:
        print("HubType column not found.")
        return
    
    # Create summary DataFrame
    summary_data = []
    
    for hub_type in df['HubType'].unique():
        type_df = df[df['HubType'] == hub_type]
        
        row = {
            'Hub Type': hub_type,
            'Count': len(type_df),
            'Avg Demand': type_df['TotalDemand'].mean() if 'TotalDemand' in type_df.columns else 0,
            'Total Demand': type_df['TotalDemand'].sum() if 'TotalDemand' in type_df.columns else 0,
        }
        
        if score_col in type_df.columns:
            row['Avg Score'] = type_df[score_col].mean()
            row['Min Score'] = type_df[score_col].min()
            row['Max Score'] = type_df[score_col].max()
        
        summary_data.append(row)
    
    summary_df = pd.DataFrame(summary_data)
    
    # Format numeric columns
    if 'Avg Demand' in summary_df.columns:
        summary_df['Avg Demand'] = summary_df['Avg Demand'].apply(lambda x: f"{x:,.0f}")
    if 'Total Demand' in summary_df.columns:
        summary_df['Total Demand'] = summary_df['Total Demand'].apply(lambda x: f"{x:,.0f}")
    if 'Avg Score' in summary_df.columns:
        summary_df['Avg Score'] = summary_df['Avg Score'].apply(lambda x: f"{x:.3f}")
    if 'Min Score' in summary_df.columns:
        summary_df['Min Score'] = summary_df['Min Score'].apply(lambda x: f"{x:.3f}")
    if 'Max Score' in summary_df.columns:
        summary_df['Max Score'] = summary_df['Max Score'].apply(lambda x: f"{x:.3f}")
    
    display(summary_df)
    
    # Total count
    print(f"\nTotal hubs: {len(df)}")

display_summary_statistics(df)

## Step 7: Export Results

Save the enriched data with hub names to a new CSV file.

In [ ]:
# Export to CSV with hub names included
output_file = RESULTS_DIR / f"hubs_with_names_{pd.Timestamp.now().strftime('%Y%m%d')}.csv"

# Reorder columns to put important ones first
priority_cols = ['group', 'HubName', 'HubType', 'Overall_Rank', 'Rank_within_HubType', 
                 'TotalDemand', score_col, 'address', 'area', 'Num_Modes']
other_cols = [c for c in df.columns if c not in priority_cols]
ordered_cols = [c for c in priority_cols if c in df.columns] + other_cols

df_export = df[ordered_cols].sort_values('Overall_Rank')
df_export.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Results exported to: {output_file}")
print(f"Total rows: {len(df_export)}")
print(f"Total columns: {len(df_export.columns)}")

## Summary

Post-processing complete:

1. **HubName column added** - Names merged from external CSV or derived from address
2. **Tables displayed**:
   - Overall top hubs ranking
   - Separate tables by hub type (ארצי, מטרופוליני, עירוני, etc.)
   - Summary statistics
3. **Results exported** - CSV file with hub names and rankings